# SPARE report validation companion

## tl;dr

This notebook reads the same local authoritative artifacts as the technical report and checks the central data, localization, force, and release-status claims. Outputs were executed and embedded when the report was built.


In [1]:
from pathlib import Path
import json
import pandas as pd

WORKSPACE = Path('C:\\Users\\DLSU\\OneDrive\\Documents\\SOFTWARE CALIBRATION')
PROJECT = WORKSPACE / 'calibration_gui'
MODEL = PROJECT / 'models' / 'live_sensor_experimental_hybrid_v4'
CHAR = PROJECT / 'analysis_outputs' / 'live_sensor_study' / 'characterization' / 'evidence-products-manual-first-authoritative'
FEATURE = PROJECT / 'analysis_outputs' / 'live_sensor_study' / 'feature_store' / 'feature-store-c0ec762f1dc888a7'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

metrics = read_json(MODEL / 'metrics.json')
release = read_json(MODEL / 'release_decision.json')
reconciliation = read_json(FEATURE / 'reconciliation_report.json')
metric_status = pd.read_csv(CHAR / 'tables' / 'metric_status.csv')
print('Workspace:', WORKSPACE)
print('Artifacts loaded:', MODEL.name, CHAR.name, FEATURE.name)


Workspace: C:\Users\DLSU\OneDrive\Documents\SOFTWARE CALIBRATION
Artifacts loaded: live_sensor_experimental_hybrid_v4 evidence-products-manual-first-authoritative feature-store-c0ec762f1dc888a7


## Context & Methods

The application model is manual-only and camera-only. The automatic archive is allowed only for hysteresis/creep-related characterization. Independent sessions, not frames, define replication. The six TEST-group folds are post-hoc because they had been inspected during recovery.


In [2]:
assert release['status'] == 'experimental'
assert release['validated_claim_allowed'] is False
assert release['force_resolution_established'] is False
print('Release status:', release['status'])
print('Validated claim allowed:', release['validated_claim_allowed'])
print('Force resolution established:', release['force_resolution_established'])
print('Blocking reasons:')
for reason in release['blocking_reasons']:
    print('-', reason)


Release status: experimental
Validated claim allowed: False
Force resolution established: False
Blocking reasons:
- The six existing outer TEST folds were reused after inspection; no untouched confirmation data remain.
- The force estimator failed the combined improvement/rank/response-resolution gate.
- The authoritative all-ROI common safe-range procedure remains failed.
- Known-force physical, latency/soak, and representative-user validation were not performed.


## Data

The immutable feature store reconciles the complete decoded video population.


In [3]:
assert reconciliation['session_count'] == 245
assert reconciliation['total_rows'] == 179176
assert reconciliation['missing_video_frames'] == 0
assert reconciliation['unmatched_video_frames'] == 0
assert len(metric_status) == 18
print('Sessions:', reconciliation['session_count'])
print('Decoded frames:', reconciliation['total_rows'])
print('Missing/unmatched:', reconciliation['missing_video_frames'], reconciliation['unmatched_video_frames'])
print('Characterization metric-status rows:', len(metric_status))


Sessions: 245
Decoded frames: 179176
Missing/unmatched: 0 0
Characterization metric-status rows: 18


## Results

Headline results are printed with the claim-limiting force diagnostics.


In [4]:
a = metrics['aggregate']
print(f"Localization macro F1: {a['event_localization_macro_f1']:.4%}")
print(f"Localization accuracy: {a['event_localization_accuracy']:.4%}")
print('Events / sessions / folds:', a['event_count'], a['independent_session_count'], a['fold_count'])
print(f"Selective F1 / coverage: {a['selective_localization_macro_f1']:.4%} / {a['selective_localization_coverage']:.4%}")
print(f"Conditional force MAE: {a['mean_conditional_force_mae_N']:.6f} N")
print(f"p95 absolute error: {a['cross_validated_p95_absolute_error_N']:.6f} N")
print(f"Mean force Spearman rho: {a['mean_force_spearman_rho']:.6f}")
print(f"Expanded gain over constant: {a['expanded_force_search_relative_gain_over_constant']:.4%}")
assert a['event_count'] == 102 and a['independent_session_count'] == 54 and a['fold_count'] == 6
assert abs(a['event_localization_macro_f1'] - 0.9342785558588881) < 1e-12
assert abs(a['mean_conditional_force_mae_N'] - 0.32798681555113973) < 1e-12


Localization macro F1: 93.4279%
Localization accuracy: 93.4117%
Events / sessions / folds: 102 54 6
Selective F1 / coverage: 100.0000% / 82.9274%
Conditional force MAE: 0.327987 N
p95 absolute error: 0.642852 N
Mean force Spearman rho: 0.206775
Expanded gain over constant: 4.4936%


## Live Sensor GUI contract

The tenth application tab is a guarded camera-only presentation with an explicit warning, unloaded warm-up, replay, event capture, and post-processing export.


In [5]:
ui_source = (PROJECT / 'gui' / 'live_sensor_tab.py').read_text(encoding='utf-8')
report_source = (PROJECT / 'core' / 'live_sensor_report.py').read_text(encoding='utf-8')
spec = read_json(PROJECT / 'config' / 'manual_only_experimental_hybrid_v4.json')
assert 'EXPERIMENTAL — NOT VALIDATED' in ui_source
assert 'Unloaded warm-up: %v/%m frames' in ui_source
assert 'Export post-processing report' in ui_source
assert 'post_processing_report.html' in report_source
assert spec['deployment_inputs']['load_cell'] == 'forbidden'
assert spec['deployment_inputs']['printer'] == 'forbidden'
print('Sensor mode:', spec['sensor_mode'])
print('GUI safeguards: warning + 120-frame warm-up + replay + HTML/CSV/JSON export')
print('Runtime load cell/printer:', spec['deployment_inputs']['load_cell'], '/', spec['deployment_inputs']['printer'])


Sensor mode: hybrid_force_event
GUI safeguards: warning + 120-frame warm-up + replay + HTML/CSV/JSON export
Runtime load cell/printer: forbidden / forbidden


## Takeaways

The artifact checks support a physically exercised acquisition/calibration program and strong retrospective discrete localization. They do not support a validated force sensor, force resolution, extrapolation, true multi-contact force, or prospective model claim.
